In [ ]:
#data analysis libraries 
import numpy as np
import pandas as pd

#visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

#ignore warnings
import warnings
warnings.filterwarnings('ignore')


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
#import train and test CSV files
train = pd.read_csv("/kaggle/input/titanic/train.csv")
test = pd.read_csv("/kaggle/input/titanic/test.csv")
PassengerId = test.PassengerId

In [ ]:
def data_missing(df):
    missing_percent = (df.isnull().sum() / len(df)) * 100
    missing_count = pd.isnull(train).sum()
    missing_info = pd.DataFrame({
        'Missing Values': missing_count,
        'Missing Percentage': missing_percent
    })
    print(missing_info)

def print_line(topic):
    print("\n************************************************")
    print(topic)
    print("************************************************\n")

In [ ]:
print_line("All available features(col) name")
print(train.columns)

print_line("Missing info for train DataFrame")
print(data_missing(train))

print_line("Missing info for test DataFrame")
print(data_missing(test))

print_line("Sample of the dataset to get an idea of the variables")
print(train.sample(5))

print_line("Data types")
print(train.dtypes)

print_line("Summary of the dataset")
train.describe(include = "all")

In [ ]:
def data_visual_barchart(x_axis_feature, y_axis_feature, df):
    plt.figure(figsize=(10, 6)) 
    sns.barplot(
        x = x_axis_feature, 
        y = y_axis_feature, 
        data=df,
        palette="viridis"
    )
    plt.title(f'Bar Chart of {feature} vs {target}')
    plt.xlabel(feature)
    plt.ylabel(target)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

In [ ]:
target = "Survived"
for feature in train.columns:
    unique_count = train[feature].nunique()
    if feature != target:
        print("Numbers of unique value(s) for", feature, "is", train[feature].nunique())
        #data_visual_barchart(feature, target, train)

In [ ]:
# fillup missing values

train.Fare.fillna(train.Fare.median(), inplace=True)
train.Embarked.fillna('S')
train.Age.fillna(train.Age.median(), inplace=True)
train.Sex = pd.factorize(train.Sex)[0]
train.Name = pd.factorize(train.Name)[0]
train.Ticket = pd.factorize(train.Ticket)[0]
train.Cabin = pd.factorize(train.Cabin)[0]
train.Embarked = pd.factorize(train.Embarked)[0]


test.Fare.fillna(test.Fare.median(), inplace=True)
test.Embarked.fillna(test.Embarked.mode()[0], inplace=True)
test.Age.fillna(test.Age.median(), inplace=True)
test.Sex = pd.factorize(test.Sex)[0]
test.Name = pd.factorize(test.Name)[0]
test.Ticket = pd.factorize(test.Ticket)[0]
test.Cabin = pd.factorize(test.Cabin)[0]
test.Embarked = pd.factorize(test.Embarked)[0]

In [ ]:
# is alone
#train['IsAlone'] = (train['SibSp'] + train['Parch']) == True
#test['IsAlone'] = (test['SibSp'] + test['Parch']) == True

# is SibSp
#train['IsSibSp'] = train['SibSp'] == True
#test['IsSibSp'] = test['SibSp'] == True

# is Parch
#train['IsParch'] = train['Parch'] == True
#test['IsParch'] = test['Parch'] == True

# FamilySize
train['FamilySize'] = train['SibSp'] + train['Parch'] + 1
test['FamilySize'] = test['SibSp'] + test['Parch'] + 1

# Cabin
train["CabinBool"] = (train["Cabin"].notnull().astype('int'))
test["CabinBool"] = (test["Cabin"].notnull().astype('int'))

# Define the bins (age ranges) and corresponding labels for the age groups
bins = [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, float('inf')]
labels = ['Child1', 'Child2', 'Child3', 'Child4', 'Young Adult1', 'Young Adult2', 'Young Adult3', 'Adult1', 'Adult2', 'Adult3', 'Middle-aged', 'Senior1', 'Senior2', 'Senior3', 'Senior4']

# Create the 'Age Group' column using pd.cut()
train['AgeGroup'] = pd.cut(train['Age'], bins=bins, labels=labels, right=False)
test['AgeGroup'] = pd.cut(test['Age'], bins=bins, labels=labels, right=False)
train.AgeGroup = pd.factorize(train.AgeGroup)[0].astype('int')
test.AgeGroup = pd.factorize(test.AgeGroup)[0].astype('int')

fare_bins = [0, 50, np.inf]
fare_labels = [1, 2]

train['FareGroup'] = pd.cut(train['Fare'], bins=fare_bins, labels=fare_labels, include_lowest=True, right=False).astype('float')
test['FareGroup'] = pd.cut(test['Fare'], bins=fare_bins, labels=fare_labels, include_lowest=True, right=False).astype('float')

print_line("Missing info for train DataFrame")
print(data_missing(train))

print_line("Missing info for test DataFrame")
print(data_missing(test))


print("Feature engineering done.....")

In [ ]:
target = "Survived"
for feature in train.columns:
    unique_count = train[feature].nunique()
    if feature != target:
        print("Numbers of unique value(s) for", feature, "is", train[feature].nunique())
        #data_visual_barchart(feature, target, train)

In [ ]:
def data_visual_scatter_plot(x_axis_feature, y_axis_feature, hue_feature):
    plt.figure(figsize=(10, 6)) 
    print(x_axis_feature.head())
    sns.scatterplot(
        x = x_axis_feature, 
        y = y_axis_feature, 
        hue = hue_feature
    )
data_visual_scatter_plot(train['FareGroup'], train['AgeGroup'], train['Survived'])

In [ ]:
print(train.dtypes)
#['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked', 'FamilySize', 'CabinBool', 'AgeGroup', 'FareGroup']
drop_col = ['PassengerId', 'Survived', 'Name', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin']
drop_col_test = ['PassengerId', 'Name', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin']

In [ ]:
# test models
from sklearn.model_selection import train_test_split

PassengerId = test.PassengerId
features = train.drop(drop_col, axis=1)
target = train["Survived"]
X_test = test.drop(drop_col_test, axis=1)

print(train.dtypes)

x_train, x_val, y_train, y_val = train_test_split(features, target, test_size = 0.22, random_state = 0)

In [ ]:
# Gaussian Naive Bayes
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

gaussian = GaussianNB()
gaussian.fit(x_train, y_train)
y_pred = gaussian.predict(x_val)
acc_gaussian = round(accuracy_score(y_pred, y_val) * 100, 2)
print(acc_gaussian)

In [ ]:
# Logistic Regression
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression()
logreg.fit(x_train, y_train)
y_pred = logreg.predict(x_val)
acc_logreg = round(accuracy_score(y_pred, y_val) * 100, 2)
print(acc_logreg)


In [ ]:
# Support Vector Machines
from sklearn.svm import SVC

svc = SVC()
svc.fit(x_train, y_train)
y_pred = svc.predict(x_val)
acc_svc = round(accuracy_score(y_pred, y_val) * 100, 2)
print(acc_svc)

In [ ]:
# Linear SVC
from sklearn.svm import LinearSVC

linear_svc = LinearSVC()
linear_svc.fit(x_train, y_train)
y_pred = linear_svc.predict(x_val)
acc_linear_svc = round(accuracy_score(y_pred, y_val) * 100, 2)
print(acc_linear_svc)

In [ ]:
# Perceptron
from sklearn.linear_model import Perceptron

perceptron = Perceptron()
perceptron.fit(x_train, y_train)
y_pred = perceptron.predict(x_val)
acc_perceptron = round(accuracy_score(y_pred, y_val) * 100, 2)
print(acc_perceptron)

In [ ]:
#Decision Tree
from sklearn.tree import DecisionTreeClassifier

decisiontree = DecisionTreeClassifier()
decisiontree.fit(x_train, y_train)
y_pred = decisiontree.predict(x_val)
acc_decisiontree = round(accuracy_score(y_pred, y_val) * 100, 2)
print(acc_decisiontree)

In [ ]:
# Random Forest
from sklearn.ensemble import RandomForestClassifier

randomforest = RandomForestClassifier()
randomforest.fit(x_train, y_train)
y_pred = randomforest.predict(x_val)
acc_randomforest = round(accuracy_score(y_pred, y_val) * 100, 2)
print(acc_randomforest)

In [ ]:
# KNN or k-Nearest Neighbors
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier()
knn.fit(x_train, y_train)
y_pred = knn.predict(x_val)
acc_knn = round(accuracy_score(y_pred, y_val) * 100, 2)
print(acc_knn)

In [ ]:
# Stochastic Gradient Descent
from sklearn.linear_model import SGDClassifier

sgd = SGDClassifier()
sgd.fit(x_train, y_train)
y_pred = sgd.predict(x_val)
acc_sgd = round(accuracy_score(y_pred, y_val) * 100, 2)
print(acc_sgd)

In [ ]:
# Gradient Boosting Classifier
from sklearn.ensemble import GradientBoostingClassifier

gbk = GradientBoostingClassifier()
gbk.fit(x_train, y_train)
y_pred = gbk.predict(x_val)
acc_gbk = round(accuracy_score(y_pred, y_val) * 100, 2)
print(acc_gbk)

In [ ]:
# cat Boost Classifier
from catboost import CatBoostClassifier

cat = CatBoostClassifier(iterations=500, learning_rate=0.01, verbose=0, random_state=42)
cat.fit(x_train, y_train)
y_pred = cat.predict(x_val)
acc_cat = round(accuracy_score(y_pred, y_val) * 100, 2)
print(acc_cat)


In [ ]:
models = pd.DataFrame({
    'Model': ['Support Vector Machines', 'KNN', 'Logistic Regression', 
              'Random Forest', 'Naive Bayes', 'Perceptron', 'Linear SVC', 
              'Decision Tree', 'Stochastic Gradient Descent', 'Gradient Boosting Classifier', 'Cat Boost Classifier'],
    'Score': [acc_svc, acc_knn, acc_logreg, 
              acc_randomforest, acc_gaussian, acc_perceptron,acc_linear_svc, acc_decisiontree,
              acc_sgd, acc_gbk, acc_cat]})
models.sort_values(by='Score', ascending=False)

In [ ]:
from sklearn.ensemble import VotingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression

X = train.drop(drop_col, axis=1)
X_test = test.drop(drop_col_test, axis=1)
y = train['Survived']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.fit_transform(X_test)

# model
gnb = GaussianNB()
xgb = XGBClassifier(n_estimators=500, learning_rate=0.01, max_depth=3, random_state=42)
lgb = LGBMClassifier(n_estimators=500, learning_rate=0.01, random_state=42)
#svm = SVC(probability=True, random_state=42)  # probability=True for voting

# votting classifier
ensemble = VotingClassifier(
    estimators=[
        ('dt', decisiontree),
        ('rf', randomforest),
        ('cat', cat),
        ('gbk', gbk),
        ('gnb', gnb),
        ('lr', logreg),
        ('xgb', xgb),
        ('lgb', lgb),
        ('svm', SVC(probability=True, random_state=42))
    ],
    voting='soft'
)

scores = cross_val_score(ensemble, X, y, cv=5, scoring='accuracy')
print(f"CV Accuracy: {scores.mean():.4f} ± {scores.std():.4f}")
ensemble.fit(X_scaled, y)

predictions = ensemble.predict(X_test_scaled)
output = pd.DataFrame({'PassengerId': PassengerId, 'Survived': predictions})
print(output)
output.to_csv('submission.csv', index = False)
print("submission was successfully saved!")